注意：运行下面的代码首先要执行 [gen_demo_factor_data.py](./tools/gen_demo_factor_data.py) 脚本生成示例数据。

# 因子框架

## 因子开发

因子数据模型自上而下分为三层结构:
* 因子库
* 因子表：因子库包含多张因子表，可以通过因子库的 `getTable` 方法获取库中的因子表对象。每张因子表的数据逻辑上是一个三维数组, 第一维是因子名称, 第二维是时间点, 第三维是证券代码。
* 因子：每张因子表中又包含多个因子，可以通过因子表的 `getFactor` 方法获取表中的因子对象。每个因子的数据逻辑上是一个 DataFrame, index 是时间点, columns 是证券代码。

可以将算子作用在因子上以衍生新的因子。多个因子逻辑上构成一个有向无环的计算图(DAG)。

所有的因子划分成两类:
* **基础因子**: 指直接由原始数据转化而来, 并不依赖于其他因子的因子, 比如股东权益因子就是以财务报表里的股东权益合计这个数据项转化而来的基础因子。大多数的基础因子通过因子表的 `getFactor` 方法得到。
* **衍生因子**: 依赖于其他因子（下称描述子）通过因子运算得到的因子, 比如市净率因子是由总市值除以股东权益得到。

因子的运算分成四类，每种运算的适用范围各不相同, 在效率上也有所差异：
* **单点运算**: 某时点、某证券的因子值只依赖描述子同时点、同证券的运算。最简单的一类因子运算, 典型例子是各种估值因子的定义, 比如市净率(PB)因子, 其是由总市值除以股东权益得到。
* **时序运算**: 某时点、某证券的因子值依赖描述子历史时间序列、同证券的运算。时序运算通常要指定每个依赖因子的回溯期数。典型例子是移动平均线, 其是由证券过去一段时间的价格序列平均得到的。
* **截面运算**: 某时点、某证券的因子值依赖描述子同时点、其他证券的运算。典型例子是数据标准化。比如 Z-score 标准化方法, 即是用每只证券的因子值减去整个截面因子值的平均数并处以截面标准差所得。
* **面板运算**: 某时点、某证券的因子值依赖描述子历史时间序列以及其他证券的运算。这是最复杂的一种运算。典型例子是进行时间序列和截面的双重标准化。

以下是定义因子的示例

In [ ]:
# 因子定义
import numpy as np

import QuantStudio.api as QS

# 创建因子库并连接
JYDB = QS.Factor.JYDB(args={"IPAddr": "localhost", "Port": 5432, "DBName": "JYDB", "User": "postgres", "Pwd": "postgres"}).connect()

# 基础因子
FT = JYDB.getTable("资产负债表_新会计准则", args={"CalcType": "最新"})# 从因子库中获取因子表
Equity = FT.getFactor("归属母公司股东权益合计")# 从因子表中获取因子
FT = JYDB.getTable("股票行情表现", args={"LookBack": 0})
TotalCap = FT.getFactor("总市值(万元)")
Close = FT.getFactor("收盘价")

# 表达式方式定义因子, 单点运算
PB = QS.Factor.rename(TotalCap * 10000 / Equity, factor_name="PB")

# 工厂函数方式定义因子, 时序运算
def MAFunc(f, idt, iid, x, args):
    return np.mean(x[0], axis=0)
calcMA = QS.Factor.makeFactorOperator(MAFunc, operator_type="Time", args={"Arity": 1, "DTMode": "单时点", "IDMode": "多ID", "LookBack": [4]})
MA5 = calcMA(Close, factor_args={"Name": "MA5"})

# 装饰器方式定义因子, 截面运算
@QS.Factor.FactorOperatorized(operator_type="Section", args={"Arity": 1, "DTMode": "多时点"})
def calcZScore(f, idt, iid, x, args):
    return (x - np.nanmean(x, axis=1)) / np.nanstd(x, axis=1)
PB_ZScore = calcZScore(PB, factor_args={"Name": "PB_ZScore"})

# 直接实例化方式, 面板运算, 不推荐
def DoubleZScoreFunc(f, idt, iid, x, args):
    TZScore = (x[0][-1] - np.nanmean(x[0], axis=0)) / np.nanstd(x[0], axis=0)
    return (TZScore - np.nanmean(TZScore)) / np.nanstd(TZScore)
calcDoubleZScore = QS.Factor.makeFactorOperator(DoubleZScoreFunc, operator_type="Panel", args={"Arity": 1, "DTMode": "单时点"})
PB_DZScore = QS.Factor.PanelOperation(descriptors=[PB], args={"Name": "PB_DZScore", "Operator": calcDoubleZScore})